# TypedDict
 是 Python 3.8+ 引入的一种类型提示工具，即**带有类型声明的字典结构**。适合需要**快速定义字典结构**且无需 Pydantic 重量级功能的场景。
 - 这个字典应该有哪些字段
 - 每个字段的类型是什么
 - TypedDict 主要是类型声明,不是运行时强校验器。

In [2]:
from typing_extensions import TypedDict
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool  # 从 core 导入，避免 langgraph 版本冲突
from rich import print as rprint
from langchain_qwq import ChatQwen
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float
movie: MovieDict = {
    "title1": "盗梦空间", # 不会导致运行异常，但是字段不一致
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 8.8,
}
rprint(movie)

{'title1': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}

## 基本使用
### Annotated的使用
`Annotated` 用来在“类型”之外，再附加一些额外信息，即`元数据`。类似于Pydantic 的 Field 。`Annotated[类型, 附加信息1, 附加信息2, ...]`

In [ ]:
# 举例1：返回简单结构
"""
使用 TypedDict 模型定义结构化输出
"""
from typing_extensions import TypedDict, Annotated
class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影的正式名称，例如《盗梦空间》"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]
# 设置模型结构化输出
structured_llm = model.with_structured_output(MovieTypedDict)
# 调用模型并获取结构化输出
response = structured_llm.invoke("给我介绍下电影《星际穿越》")
rprint(type(response))
rprint(response)

<class 'dict'>

{'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'rating': 9.4}

In [8]:
# 举例2：返回嵌套结构
from typing import TypedDict, List, Annotated
# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]
class Movie(TypedDict):
    """电影情况"""
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "上映年份"]
    director: Annotated[str, "导演"]
    cast: Annotated[List[Actor], "演员列表"] # 嵌套列表定义
    rating: Annotated[float, "评分"]
# 设置模型结构化输出
structured_llm = model.with_structured_output(Movie)
# 调用模型并获取结构化输出
resp = structured_llm.invoke("告诉我盗梦空间的信息，不要只有一个标题")
# 访问嵌套数据
rprint(resp)

{'title': '盗梦空间'}

### ...的使用
下述代码的 ... 是Python的字面量，等价于 Ellipsis ，可以理解为占位符。
下游框架（如LangChain）可以对 ... 作定制化处理，如LangChain中Annotated的 ... 表示当前字段是必须存在的，不可省略，用来指示模型的输出。

In [10]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, ..., "导演"]
    rating: Annotated[float, ..., "电影评分，满分十分"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。")
rprint(response)
rprint(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰'}

<class 'dict'>